## RoBERTa-Tagalog Bilingual-Trained Dementia Classifier

Trains RoBERTa-Tagalog (`jcblaise/roberta-tagalog-base`) on English and Tagalog conversational transcripts
and evaluates performance on English and Tagalog test sets.

### Design Choices
- **Pooling**: We use 'masked mean pooling' across all tokens in a transcript (instead of just the special `[CLS]` token). This method helps capture the full meaning of the transcript and is consistent with how RoBERTa-Tagalog is often used for similar tasks.
- **Learning rate**: The learning rate is chosen from a lower range (between 5e-6 and 3e-5), which is typically optimal for fine-tuning the RoBERTa-Tagalog model.
- **Batch size**: We test two small batch sizes: 4 and 8. These smaller sizes are chosen because of the limited dataset size and allow for more frequent model updates during each training epoch.
- **Scheduler**: A 'Linear Warmup with Decay' schedule is used for the learning rate. This is a common and effective practice for fine-tuning models from the HuggingFace library.
- **Evaluation**: We perform a 'Stratified 10-fold Cross-Validation'. Our primary performance metric is 'Macro-F1 Score', and we also specifically track 'Recall' for the dementia class (which is also known as sensitivity).

In [ ]:
!pip install transformers==4.44.0 torch scikit-learn accelerate sentencepiece xformers -q

In [ ]:
import gc
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os

from tqdm import tqdm
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

SEEDS         = [42]
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME    = "jcblaise/roberta-tagalog-base"
MAX_LEN       = 128
MAX_EPOCHS    = 10
PATIENCE      = 3
NO_DECAY      = ["bias", "LayerNorm.weight"]
OUTPUT_DIR    = "."  # change to an absolute path when needed (e.g. your Drive folder)

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

In [ ]:
# Place english.csv and filipino.csv in the same directory as this notebook,
# or update these paths to point to your data files.
df_en = pd.read_csv("english.csv", header=None)
df_en.columns = ["text", "label"]
df_en["text"]  = df_en["text"].fillna("").astype(str)
df_en["label"] = pd.to_numeric(df_en["label"], errors='coerce').fillna(0).astype(int)

print("English class balance:")
display(df_en['label'].value_counts().sort_index().to_frame())
print(f"English total: {len(df_en)} | 0: {(df_en['label']==0).sum()} | 1: {(df_en['label']==1).sum()}")

df_tl = pd.read_csv("filipino.csv", header=None)
df_tl.columns = ["text", "label"]
df_tl["text"]  = df_tl["text"].fillna("").astype(str)
df_tl["label"] = pd.to_numeric(df_tl["label"], errors='coerce').fillna(0).astype(int)

print("\nFilipino class balance:")
display(df_tl['label'].value_counts().sort_index().to_frame())
print(f"Filipino total: {len(df_tl)} | 0: {(df_tl['label']==0).sum()} | 1: {(df_tl['label']==1).sum()}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class DementiaDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts     = df["text"].tolist()
        self.labels    = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len   = MAX_LEN

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }


class RoBERTaTagalogClassifier(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone   = AutoModel.from_pretrained(model_name)
        self.dropout    = nn.Dropout(0.1)
        self.classifier = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        hidden  = outputs.last_hidden_state
        # Masked mean pooling
        mask    = attention_mask.unsqueeze(-1).expand(hidden.size()).float()
        pooled  = (hidden * mask).sum(1) / (mask.sum(1) + 1e-6)
        pooled  = self.dropout(pooled)
        return self.classifier(pooled)


def make_optimizer(model, lr, wd):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if any(nd in n for nd in NO_DECAY):
            no_decay.append(p)
        else:
            decay.append(p)
    return torch.optim.AdamW(
        [{"params": decay, "weight_decay": wd},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=lr, betas=(0.9, 0.999), eps=1e-8,
    )


def train_epoch(model, loader, optimizer, criterion, scheduler=None):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc="Train", leave=False):
        optimizer.zero_grad()

        logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device)
        )

        if not torch.isfinite(logits).all():
            print("NaN in logits — skipping batch")
            continue

        loss = criterion(logits, batch["labels"].to(device))

        if not torch.isfinite(loss):
            print("NaN loss — skipping batch")
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item()

        all_preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())
        all_labels.extend(batch["labels"].tolist())

    train_acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), train_acc


@torch.no_grad()
def evaluate(model, loader, desc="eval"):
    model.eval()
    preds, labels = [], []

    for batch in tqdm(loader, desc=desc, leave=False):
        logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device)
        )
        if not torch.isfinite(logits).all():
            return float("nan"), float("nan"), float("nan"), float("nan"), \
                   np.array([]), np.array([]), np.array([])
        preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())
        labels.extend(batch["labels"].tolist())

    acc      = accuracy_score(labels, preds)
    f1       = f1_score(labels, preds, average="macro", zero_division=0)
    prec     = precision_score(labels, preds, average="macro", zero_division=0)
    rec      = recall_score(labels, preds, average="macro", zero_division=0)
    f1_per   = f1_score(labels, preds, average=None, zero_division=0)
    prec_per = precision_score(labels, preds, average=None, zero_division=0)
    rec_per  = recall_score(labels, preds, average=None, zero_division=0)

    return acc, f1, prec, rec, f1_per, prec_per, rec_per


@torch.no_grad()
def collect_predictions(model, df):
    """Returns df with pred, prob_HC, prob_AD, confidence, correct columns."""
    model.eval()
    all_probs = []
    loader = DataLoader(DementiaDataset(df.reset_index(drop=True), tokenizer), batch_size=16)
    for batch in loader:
        logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        all_probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
    probs             = np.vstack(all_probs)
    out               = df.reset_index(drop=True).copy()
    out["pred"]       = probs.argmax(axis=1)
    out["prob_HC"]    = probs[:, 0]
    out["prob_AD"]    = probs[:, 1]
    out["confidence"] = probs.max(axis=1)
    out["correct"]    = out["label"] == out["pred"]
    return out

In [ ]:
print("=" * 60)
print("GRID SEARCH — BILINGUAL (70% Train / 15% Val / 15% Test)")
print("=" * 60)

# ── Quick-test override: comment out the 3 lines below to run the full grid ──
# Full search: BATCH_SIZES=[4,8], LR_CANDIDATES=[5e-6,6e-6,1e-5,2e-5,3e-5], WEIGHT_DECAYS=[1e-2,1e-5]
BATCH_SIZES   = [4, 8]
LR_CANDIDATES = [5e-6, 6e-6, 1e-5, 2e-5, 3e-5]
WEIGHT_DECAYS = [1e-2, 1e-5]
# ────────────────────────────────────────────────────────────────────────────

gs_results = {}

# Add language markers so subsets can be recovered in CV and error analysis
df_en_bi = df_en.copy(); df_en_bi["lang"] = "en"
df_tl_bi = df_tl.copy(); df_tl_bi["lang"] = "tl"
df_gs_combined = pd.concat([df_en_bi, df_tl_bi], ignore_index=True)

# Create a 70/15/15 train/val/test split for hyperparameter tuning
# DementiaDataset ignores the 'lang' column (reads only 'text' and 'label')
initial_train_val_df, gs_test_df = train_test_split(df_gs_combined, test_size=0.15, stratify=df_gs_combined["label"], random_state=42)
gs_train_df, gs_val_df = train_test_split(initial_train_val_df, test_size=(0.15/0.85), stratify=initial_train_val_df["label"], random_state=42)


for bs in BATCH_SIZES:
    for lr in LR_CANDIDATES:
        for wd in WEIGHT_DECAYS:
            print(f"\nEvaluating HP combo: bs={bs}, lr={lr:.0e}, wd={wd:.0e}")
            clear_gpu()
            set_seed(42)

            train_loader = DataLoader(DementiaDataset(gs_train_df, tokenizer), batch_size=bs, shuffle=True)
            val_loader   = DataLoader(DementiaDataset(gs_val_df,   tokenizer), batch_size=bs)
            test_loader  = DataLoader(DementiaDataset(gs_test_df,  tokenizer), batch_size=bs)

            model     = RoBERTaTagalogClassifier(MODEL_NAME).to(device)
            optimizer = make_optimizer(model, lr, wd)
            criterion = nn.CrossEntropyLoss()

            num_training_steps = len(train_loader) * MAX_EPOCHS
            num_warmup_steps   = int(0.1 * num_training_steps)
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=num_warmup_steps,
                num_training_steps=num_training_steps
            )

            # Early stopping and HP selection both use val_f1, keeping gs_test_df clean.
            best_val_f1, patience_ctr, best_test_f1 = -1.0, 0, -1.0

            try:
                for epoch in range(MAX_EPOCHS):
                    loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, scheduler)
                    if not np.isfinite(loss):
                        print(f"  Epoch {epoch+1}: NaN loss, skipping combo.")
                        best_test_f1 = float("-inf")
                        break

                    val_acc,  val_f1,  _, _, _, _, _ = evaluate(model, val_loader,  desc="  Val")
                    test_acc, test_f1, _, _, _, _, _ = evaluate(model, test_loader, desc="  Test")

                    if not np.isfinite(val_f1) or not np.isfinite(test_f1):
                        print(f"  Epoch {epoch+1}: NaN metric, skipping combo.")
                        best_test_f1 = float("-inf")
                        break

                    print(f"  Epoch {epoch+1} loss={loss:.4f} train_acc={train_acc:.3f} val_acc={val_acc:.3f} val_f1={val_f1:.3f} test_acc={test_acc:.3f} test_f1={test_f1:.3f}")

                    best_test_f1 = max(best_test_f1, test_f1)

                    if val_f1 > best_val_f1:
                        best_val_f1, patience_ctr = val_f1, 0
                    else:
                        patience_ctr += 1
                        if patience_ctr >= PATIENCE:
                            print(f"  Early stopping at epoch {epoch+1}.")
                            break
            finally:
                del model, optimizer, criterion, scheduler, train_loader, val_loader, test_loader
                clear_gpu()

            gs_results[(bs, lr, wd)] = best_val_f1

BEST_BS, BEST_LR, BEST_WD = max(gs_results, key=gs_results.get)
print(f"\nBest params: bs={BEST_BS}, lr={BEST_LR:.0e}, wd={BEST_WD:.0e}, val F1={gs_results[(BEST_BS, BEST_LR, BEST_WD)]:.3f}")


In [ ]:
print("=" * 60)
print("10-FOLD CV — BILINGUAL-TRAINED MODEL")
print("Train: ~90% EN+TL combined | Test: EN subset + TL subset + combined fold")
print("=" * 60)

import pickle

# Update CHECKPOINT_DIR if not running on Colab
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, "roberta_tagalog_bilingual_cv_checkpoint.pkl")

N_SPLITS = 10

# Per-language subset metrics (each sample seen exactly once in its held-out fold)
en_sub_accs, en_sub_f1s, en_sub_precs, en_sub_recs = [], [], [], []
tl_sub_accs, tl_sub_f1s, tl_sub_precs, tl_sub_recs = [], [], [], []
both_accs, both_f1s, both_precs, both_recs = [], [], [], []
en_sub_f1_0, en_sub_f1_1, en_sub_rec_1, en_sub_prec_1 = [], [], [], []
tl_sub_f1_0, tl_sub_f1_1, tl_sub_rec_1, tl_sub_prec_1 = [], [], [], []
both_f1_0, both_f1_1, both_rec_1, both_prec_1 = [], [], [], []

# Per-sample predictions for error analysis (each sample seen exactly once)
en_cv_preds_list = []  # EN subset predictions from each fold
tl_cv_preds_list = []  # TL subset predictions from each fold

completed_seeds = set()
if os.path.exists(CHECKPOINT_FILE):
    print(f"Loading checkpoint from {CHECKPOINT_FILE}...")
    with open(CHECKPOINT_FILE, "rb") as f:
        checkpoint_data = pickle.load(f)
        en_sub_accs = checkpoint_data.get("en_sub_accs", [])
        en_sub_f1s = checkpoint_data.get("en_sub_f1s", [])
        en_sub_precs = checkpoint_data.get("en_sub_precs", [])
        en_sub_recs = checkpoint_data.get("en_sub_recs", [])
        tl_sub_accs = checkpoint_data.get("tl_sub_accs", [])
        tl_sub_f1s = checkpoint_data.get("tl_sub_f1s", [])
        tl_sub_precs = checkpoint_data.get("tl_sub_precs", [])
        tl_sub_recs = checkpoint_data.get("tl_sub_recs", [])
        both_accs = checkpoint_data.get("both_accs", [])
        both_f1s = checkpoint_data.get("both_f1s", [])
        both_precs = checkpoint_data.get("both_precs", [])
        both_recs = checkpoint_data.get("both_recs", [])
        en_sub_f1_0 = checkpoint_data.get("en_sub_f1_0", [])
        en_sub_f1_1 = checkpoint_data.get("en_sub_f1_1", [])
        en_sub_rec_1 = checkpoint_data.get("en_sub_rec_1", [])
        en_sub_prec_1 = checkpoint_data.get("en_sub_prec_1", [])
        tl_sub_f1_0 = checkpoint_data.get("tl_sub_f1_0", [])
        tl_sub_f1_1 = checkpoint_data.get("tl_sub_f1_1", [])
        tl_sub_rec_1 = checkpoint_data.get("tl_sub_rec_1", [])
        tl_sub_prec_1 = checkpoint_data.get("tl_sub_prec_1", [])
        both_f1_0 = checkpoint_data.get("both_f1_0", [])
        both_f1_1 = checkpoint_data.get("both_f1_1", [])
        both_rec_1 = checkpoint_data.get("both_rec_1", [])
        both_prec_1 = checkpoint_data.get("both_prec_1", [])
        completed_seeds = checkpoint_data.get("completed_seeds", set())
    seed_str = "seed" if len(completed_seeds) == 1 else "seeds"
    print(f"Loaded results for {len(completed_seeds)} {seed_str}: {completed_seeds}")

for seed in [s for s in SEEDS if s not in completed_seeds]:
    print(f"\n{'='*20} Seed: {seed} {'='*20}")
    set_seed(seed)
    clear_gpu()

    seed_en_sub_accs, seed_en_sub_f1s, seed_en_sub_precs, seed_en_sub_recs = [], [], [], []
    seed_tl_sub_accs, seed_tl_sub_f1s, seed_tl_sub_precs, seed_tl_sub_recs = [], [], [], []
    seed_both_accs, seed_both_f1s, seed_both_precs, seed_both_recs = [], [], [], []
    seed_en_sub_f1_0, seed_en_sub_f1_1, seed_en_sub_rec_1, seed_en_sub_prec_1 = [], [], [], []
    seed_tl_sub_f1_0, seed_tl_sub_f1_1, seed_tl_sub_rec_1, seed_tl_sub_prec_1 = [], [], [], []
    seed_both_f1_0, seed_both_f1_1, seed_both_rec_1, seed_both_prec_1 = [], [], [], []

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    for fold, (train_idx, test_idx) in enumerate(skf.split(df_gs_combined["text"], df_gs_combined["label"])):
        print(f"\n-- Seed {seed} | Fold {fold+1}/{N_SPLITS} --")
        clear_gpu()
        fold_seed = seed + fold
        set_seed(fold_seed)

        train_df = df_gs_combined.iloc[train_idx].reset_index(drop=True)

        print(f"  Training with best HPs: bs={BEST_BS}, lr={BEST_LR:.0e}, wd={BEST_WD:.0e}")
        set_seed(fold_seed + 200)
        clear_gpu()

        train_loader = DataLoader(DementiaDataset(train_df, tokenizer), batch_size=BEST_BS, shuffle=True)

        model     = RoBERTaTagalogClassifier(MODEL_NAME).to(device)
        optimizer = make_optimizer(model, BEST_LR, BEST_WD)
        criterion = nn.CrossEntropyLoss()

        num_training_steps = len(train_loader) * MAX_EPOCHS
        num_warmup_steps   = int(0.1 * num_training_steps)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=num_warmup_steps,
            num_training_steps=num_training_steps
        )

        try:
            for epoch in range(MAX_EPOCHS):
                loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, scheduler)
                if not np.isfinite(loss):
                    print(f"    Epoch {epoch+1}: NaN loss, stopping.")
                    break
                print(f"    Epoch {epoch+1} loss={loss:.4f} train_acc={train_acc:.3f}")
        finally:
            del optimizer, criterion, scheduler
            clear_gpu()

        print(f"  Evaluating Fold {fold+1}...")
        model.eval()

        # Split held-out fold into EN and TL subsets
        test_fold_df   = df_gs_combined.iloc[test_idx].reset_index(drop=True)
        en_test_subset = test_fold_df[test_fold_df["lang"] == "en"].reset_index(drop=True)
        tl_test_subset = test_fold_df[test_fold_df["lang"] == "tl"].reset_index(drop=True)

        test_en_loader   = DataLoader(DementiaDataset(en_test_subset, tokenizer), batch_size=BEST_BS)
        test_tl_loader   = DataLoader(DementiaDataset(tl_test_subset, tokenizer), batch_size=BEST_BS)
        test_both_loader = DataLoader(DementiaDataset(test_fold_df,   tokenizer), batch_size=BEST_BS)

        acc, f1, prec, rec, f1_per, prec_per, rec_per = evaluate(model, test_en_loader,   desc="    Test EN Sub")
        print(f"      English Subset — Acc: {acc:.3f}, F1: {f1:.3f}")
        seed_en_sub_accs.append(acc); seed_en_sub_f1s.append(f1); seed_en_sub_precs.append(prec); seed_en_sub_recs.append(rec)
        seed_en_sub_f1_0.append(f1_per[0]); seed_en_sub_f1_1.append(f1_per[1]); seed_en_sub_rec_1.append(rec_per[1]); seed_en_sub_prec_1.append(prec_per[1])

        acc, f1, prec, rec, f1_per, prec_per, rec_per = evaluate(model, test_tl_loader,   desc="    Test TL Sub")
        print(f"      Tagalog Subset — Acc: {acc:.3f}, F1: {f1:.3f}")
        seed_tl_sub_accs.append(acc); seed_tl_sub_f1s.append(f1); seed_tl_sub_precs.append(prec); seed_tl_sub_recs.append(rec)
        seed_tl_sub_f1_0.append(f1_per[0]); seed_tl_sub_f1_1.append(f1_per[1]); seed_tl_sub_rec_1.append(rec_per[1]); seed_tl_sub_prec_1.append(prec_per[1])

        acc, f1, prec, rec, f1_per, prec_per, rec_per = evaluate(model, test_both_loader, desc="    Test Both")
        print(f"      Combined — Acc: {acc:.3f}, F1: {f1:.3f}")
        seed_both_accs.append(acc); seed_both_f1s.append(f1); seed_both_precs.append(prec); seed_both_recs.append(rec)
        seed_both_f1_0.append(f1_per[0]); seed_both_f1_1.append(f1_per[1]); seed_both_rec_1.append(rec_per[1]); seed_both_prec_1.append(prec_per[1])

        # Collect per-sample predictions for error analysis (each sample seen exactly once)
        en_cv_preds_list.append(collect_predictions(model, en_test_subset))
        tl_cv_preds_list.append(collect_predictions(model, tl_test_subset))

        del model, train_loader, test_en_loader, test_tl_loader, test_both_loader
        clear_gpu()

    if seed_en_sub_f1s:

        en_sub_accs.extend(seed_en_sub_accs);   en_sub_f1s.extend(seed_en_sub_f1s);   en_sub_precs.extend(seed_en_sub_precs);   en_sub_recs.extend(seed_en_sub_recs)
        en_sub_f1_0.extend(seed_en_sub_f1_0); en_sub_f1_1.extend(seed_en_sub_f1_1); en_sub_rec_1.extend(seed_en_sub_rec_1); en_sub_prec_1.extend(seed_en_sub_prec_1)
        tl_sub_accs.extend(seed_tl_sub_accs);   tl_sub_f1s.extend(seed_tl_sub_f1s);   tl_sub_precs.extend(seed_tl_sub_precs);   tl_sub_recs.extend(seed_tl_sub_recs)
        tl_sub_f1_0.extend(seed_tl_sub_f1_0); tl_sub_f1_1.extend(seed_tl_sub_f1_1); tl_sub_rec_1.extend(seed_tl_sub_rec_1); tl_sub_prec_1.extend(seed_tl_sub_prec_1)
        both_accs.extend(seed_both_accs); both_f1s.extend(seed_both_f1s); both_precs.extend(seed_both_precs); both_recs.extend(seed_both_recs)
        both_f1_0.extend(seed_both_f1_0); both_f1_1.extend(seed_both_f1_1); both_rec_1.extend(seed_both_rec_1); both_prec_1.extend(seed_both_prec_1)

        completed_seeds.add(seed)
        checkpoint_data = {
            "en_sub_accs": en_sub_accs, "en_sub_f1s": en_sub_f1s, "en_sub_precs": en_sub_precs, "en_sub_recs": en_sub_recs,
            "tl_sub_accs": tl_sub_accs, "tl_sub_f1s": tl_sub_f1s, "tl_sub_precs": tl_sub_precs, "tl_sub_recs": tl_sub_recs,
            "both_accs": both_accs, "both_f1s": both_f1s, "both_precs": both_precs, "both_recs": both_recs,
            "en_sub_f1_0": en_sub_f1_0, "en_sub_f1_1": en_sub_f1_1, "en_sub_rec_1": en_sub_rec_1, "en_sub_prec_1": en_sub_prec_1,
            "tl_sub_f1_0": tl_sub_f1_0, "tl_sub_f1_1": tl_sub_f1_1, "tl_sub_rec_1": tl_sub_rec_1, "tl_sub_prec_1": tl_sub_prec_1,
            "both_f1_0": both_f1_0, "both_f1_1": both_f1_1, "both_rec_1": both_rec_1, "both_prec_1": both_prec_1,
            "completed_seeds": completed_seeds
        }
        with open(CHECKPOINT_FILE, "wb") as f:
            pickle.dump(checkpoint_data, f)
        print(f"  Checkpoint saved for seed {seed}.")
    else:
        print(f"  No valid results for seed {seed}.")

    print(f"{'='*20} Finished Seed: {seed} {'='*20}")
    clear_gpu()

print("\nCV Complete.")

# Both languages: concatenate fold predictions — each sample seen exactly once
if en_cv_preds_list:
    en_cv_results = pd.concat(en_cv_preds_list).reset_index(drop=True)
    print(f"English CV predictions collected for all {len(en_cv_results)} samples.")

if tl_cv_preds_list:
    tl_cv_results = pd.concat(tl_cv_preds_list).reset_index(drop=True)
    print(f"Tagalog CV predictions collected for all {len(tl_cv_results)} samples.")


In [ ]:
def fmt(vals):
    if len(vals) == 0:
        return "nan±nan"
    vals = np.asarray(vals, dtype=float)
    return f"{np.mean(vals):.3f}±{np.std(vals):.3f}"

print("\n" + "=" * 75)
print("BILINGUAL-TRAINED ROBERTA-TAGALOG — FINAL RESULTS (mean ± std across folds)")
print("=" * 75)

print(f"\n{'Metric':<28} {'English Fold':>14} {'Tagalog Fold':>14} {'Combined':>14}")
print("-" * 70)
print(f"{'Accuracy':<28} {fmt(en_sub_accs):>14} {fmt(tl_sub_accs):>14} {fmt(both_accs):>14}")
print(f"{'F1 Macro':<28} {fmt(en_sub_f1s):>14} {fmt(tl_sub_f1s):>14} {fmt(both_f1s):>14}")
print(f"{'Precision Macro':<28} {fmt(en_sub_precs):>14} {fmt(tl_sub_precs):>14} {fmt(both_precs):>14}")
print(f"{'Recall Macro':<28} {fmt(en_sub_recs):>14} {fmt(tl_sub_recs):>14} {fmt(both_recs):>14}")
print("-" * 70)
print(f"{'F1 Healthy (class 0)':<28} {fmt(en_sub_f1_0):>14} {fmt(tl_sub_f1_0):>14} {fmt(both_f1_0):>14}")
print(f"{'F1 AD (class 1)':<28} {fmt(en_sub_f1_1):>14} {fmt(tl_sub_f1_1):>14} {fmt(both_f1_1):>14}")
print(f"{'AD Recall (sensitivity)':<28} {fmt(en_sub_rec_1):>14} {fmt(tl_sub_rec_1):>14} {fmt(both_rec_1):>14}")
print(f"{'AD Precision (PPV)':<28} {fmt(en_sub_prec_1):>14} {fmt(tl_sub_prec_1):>14} {fmt(both_prec_1):>14}")
print("-" * 70)

if len(en_sub_f1s) > 0 and len(tl_sub_f1s) > 0:
    diffs   = np.array(en_sub_f1s) - np.array(tl_sub_f1s)
    gap     = np.mean(diffs)
    gap_std = np.std(diffs)
    print(f"\nEN - TL F1 macro within bilingual fold: {gap:.3f}±{gap_std:.3f}")
else:
    print("\nEN - TL F1 macro within bilingual fold: nan±nan")


In [ ]:
print("=" * 60)
print("ERROR ANALYSIS")
print("=" * 60)

os.makedirs(OUTPUT_DIR, exist_ok=True)

def add_error_type(df):
    out = df.copy()
    out["error_type"] = out.apply(
        lambda r: "FN" if (r["label"] == 1 and r["pred"] == 0)
                  else ("FP" if (r["label"] == 0 and r["pred"] == 1) else "correct"),
        axis=1,
    )
    return out

# Both languages are unbiased (each seen exactly once) — no folds_wrong column needed
PRED_COLS = ["text", "label", "pred", "prob_HC", "prob_AD", "confidence", "error_type"]

# ── Section 1: CV-Based (both languages unbiased) ────────────────────────────────
print("\n── Section 1: CV-Based Error Analysis ──")

cv_available = "en_cv_results" in dir() and "tl_cv_results" in dir()

if cv_available:
    for lang_name, res_df, fname in [
        ("English (CV, each sample seen once)", en_cv_results, os.path.join(OUTPUT_DIR, "errors_cv_english.csv")),
        ("Tagalog (CV, each sample seen once)", tl_cv_results, os.path.join(OUTPUT_DIR, "errors_cv_tagalog.csv")),
    ]:
        res  = add_error_type(res_df)
        errs = res[res["error_type"] != "correct"].reset_index(drop=True)
        fn   = (res["error_type"] == "FN").sum()
        fp   = (res["error_type"] == "FP").sum()
        print(f"\n  ── {lang_name} ──")
        print(f"  Accuracy : {res['correct'].mean():.3f}  |  Errors: {len(errs)}/{len(res)}")
        print(f"  False Negatives (AD → Healthy) : {fn}  |  False Positives (Healthy → AD) : {fp}")
        display(errs[PRED_COLS].sort_values("confidence", ascending=False))
        errs[PRED_COLS].to_csv(fname, index=False)
        print(f"  Saved → {fname}")
else:
    print("  CV results not available — re-run the CV cell without resuming from checkpoint.")

# ── Section 2: Final Model (bilingual gs_train + gs_val → EN/TL subsets of gs_test_df) ──
print("\n── Section 2: Final Model — trained on bilingual gs_train_df (70%) + gs_val_df (15%) ──")

set_seed(42)
clear_gpu()

bi_train_final  = pd.concat([gs_train_df, gs_val_df]).reset_index(drop=True)
train_loader_ea = DataLoader(DementiaDataset(bi_train_final, tokenizer), batch_size=BEST_BS, shuffle=True)
final_model     = RoBERTaTagalogClassifier(MODEL_NAME).to(device)
opt_ea          = make_optimizer(final_model, BEST_LR, BEST_WD)
crit_ea         = nn.CrossEntropyLoss()
n_steps         = len(train_loader_ea) * MAX_EPOCHS
sched_ea        = get_linear_schedule_with_warmup(opt_ea, int(0.1 * n_steps), n_steps)

for epoch in range(MAX_EPOCHS):
    loss, acc = train_epoch(final_model, train_loader_ea, opt_ea, crit_ea, sched_ea)
    print(f"  Epoch {epoch+1}  loss={loss:.4f}  acc={acc:.3f}")

del opt_ea, crit_ea, sched_ea, train_loader_ea
clear_gpu()

_model_path = os.path.join(OUTPUT_DIR, "final_model_roberta_tagalog_bilingual.pt")
torch.save(final_model.state_dict(), _model_path)
print(f"  Saved → {{_model_path}}")

# Evaluate on EN subset, TL subset, and full held-out test fold
_gs_en = gs_test_df[gs_test_df["lang"] == "en"].reset_index(drop=True)
_gs_tl = gs_test_df[gs_test_df["lang"] == "tl"].reset_index(drop=True)

en_fm_results      = collect_predictions(final_model, _gs_en)
tl_fm_results      = collect_predictions(final_model, _gs_tl)
combined_fm_results = collect_predictions(final_model, gs_test_df)

del final_model
clear_gpu()

for split_name, res, fname in [
    ("English (gs_test_df EN subset)", en_fm_results,       os.path.join(OUTPUT_DIR, "errors_final_english.csv")),
    ("Tagalog (gs_test_df TL subset)", tl_fm_results,       os.path.join(OUTPUT_DIR, "errors_final_tagalog.csv")),
    ("Combined (full gs_test_df)",     combined_fm_results, os.path.join(OUTPUT_DIR, "errors_final_combined.csv")),
]:
    res  = add_error_type(res)
    errs = res[res["error_type"] != "correct"].reset_index(drop=True)
    fn   = (res["error_type"] == "FN").sum()
    fp   = (res["error_type"] == "FP").sum()
    print(f"\n  ── {split_name} ──")
    print(f"  Accuracy : {res['correct'].mean():.3f}  |  Errors: {len(errs)}/{len(res)}")
    print(f"  False Negatives (AD → Healthy) : {fn}  |  False Positives (Healthy → AD) : {fp}")
    if not errs.empty:
        display(errs[PRED_COLS].sort_values("confidence", ascending=False))
    errs[PRED_COLS].to_csv(fname, index=False)
    print(f"  Saved → {fname}")

# ── Save results summary ──────────────────────────────────────────────────────────────
def _ms(vals):
    v = np.asarray(vals, dtype=float)
    return round(float(np.mean(v)), 4), round(float(np.std(v)), 4)

def _final_metrics(df):
    yt, yp = df["label"].values, df["pred"].values
    f1_per   = f1_score(yt, yp, average=None, zero_division=0)
    prec_per = precision_score(yt, yp, average=None, zero_division=0)
    rec_per  = recall_score(yt, yp, average=None, zero_division=0)
    return {
        "accuracy":        round(float(accuracy_score(yt, yp)), 4),
        "f1_macro":        round(float(f1_score(yt, yp, average="macro", zero_division=0)), 4),
        "precision_macro": round(float(precision_score(yt, yp, average="macro", zero_division=0)), 4),
        "recall_macro":    round(float(recall_score(yt, yp, average="macro", zero_division=0)), 4),
        "f1_healthy":      round(float(f1_per[0]), 4),
        "f1_ad":           round(float(f1_per[1]), 4),
        "ad_recall":       round(float(rec_per[1]), 4),
        "ad_precision":    round(float(prec_per[1]), 4),
    }

_NAN = float("nan")
rows = []

# Best hyperparameters (optimised on bilingual combined data)
for _name, _val in [("batch_size", BEST_BS), ("learning_rate", BEST_LR), ("weight_decay", BEST_WD)]:
    rows.append({"section": "best_hp", "metric": _name,
                 "tagalog_mean": _NAN,  "tagalog_std": _NAN,
                 "english_mean": _NAN,  "english_std": _NAN,
                 "combined_mean": _val, "combined_std": _NAN})

# CV summary (mean ± std across folds)
for _metric, _en, _tl, _bo in [
    ("accuracy",        en_sub_accs,   tl_sub_accs,   both_accs),
    ("f1_macro",        en_sub_f1s,    tl_sub_f1s,    both_f1s),
    ("precision_macro", en_sub_precs,  tl_sub_precs,  both_precs),
    ("recall_macro",    en_sub_recs,   tl_sub_recs,   both_recs),
    ("f1_healthy",      en_sub_f1_0,   tl_sub_f1_0,   both_f1_0),
    ("f1_ad",           en_sub_f1_1,   tl_sub_f1_1,   both_f1_1),
    ("ad_recall",       en_sub_rec_1,  tl_sub_rec_1,  both_rec_1),
    ("ad_precision",    en_sub_prec_1, tl_sub_prec_1, both_prec_1),
]:
    _en_m, _en_s = _ms(_en); _tl_m, _tl_s = _ms(_tl); _bo_m, _bo_s = _ms(_bo)
    rows.append({"section": "cv", "metric": _metric,
                 "tagalog_mean": _tl_m, "tagalog_std": _tl_s,
                 "english_mean": _en_m, "english_std": _en_s,
                 "combined_mean": _bo_m, "combined_std": _bo_s})

# Final model point estimates (EN/TL subsets + combined from gs_test_df)
_en_fm = _final_metrics(en_fm_results)
_tl_fm = _final_metrics(tl_fm_results)
_bo_fm = _final_metrics(combined_fm_results)
for _metric in ["accuracy", "f1_macro", "precision_macro", "recall_macro",
                "f1_healthy", "f1_ad", "ad_recall", "ad_precision"]:
    rows.append({"section": "final_model", "metric": _metric,
                 "tagalog_mean": _tl_fm[_metric], "tagalog_std": _NAN,
                 "english_mean": _en_fm[_metric], "english_std": _NAN,
                 "combined_mean": _bo_fm[_metric], "combined_std": _NAN})

RESULTS_FILE = os.path.join(OUTPUT_DIR, "results_roberta_tagalog_bilingual.csv")
pd.DataFrame(rows).to_csv(RESULTS_FILE, index=False)
print(f"\nResults saved → {{RESULTS_FILE}}")
